# ColdChain Optimized — Module 2: Route Optimization & Backhaul Matching

**Extends the dynamic pricing notebook with two new capabilities:**
- **Route Optimization** — given origin + destination, rank all viable route options by time, cost, and a composite efficiency score
- **Backhaul Matching** — after a delivery, find return-leg cargo opportunities to eliminate empty running

---
### How these connect to the pricing model
- Route options are priced using `compute_trip_price()` from Module 1
- Backhaul matches use `proportional_split()` / `shapley_approx_split()` for cost allocation
- Both outputs feed the same KPI table: Cost per Ton-Km, Shipper Cost Savings %, Margin Uplift %

---
## 1. Setup — imports and data loading

In [ ]:
import json, re, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from itertools import combinations
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

# ── File paths ────────────────────────────────────────────────────────────────
TRIPS_PATH  = 'Delivery_truck_trip_data.csv'
TOLLS_PATH  = 'tolls-with-metadata.csv'
DIESEL_PATH = 'all_india_live_diesel.csv'

# ── Pricing model parameters (from Module 1) ──────────────────────────────────
BEST_PARAMS = {
    'fuel_eff':            0.35,
    'refrig_rs_hr':        180,
    'avg_speed_kmh':       50,
    'driver_cost_per_day': 3000,
    'km_per_driver_day':   400,
    'truck_capacity_tons': 20,
    'demand_weight':       0.15,
    'dwell_penalty_rate':  0.01,
    'free_dwell_min':      60,
    'margin':              0.18,
}

# ── Efficiency score weights (tune for your use case) ─────────────────────────
EFFICIENCY_WEIGHTS = {
    'cost':  0.45,   # lower cost = better
    'time':  0.35,   # lower time = better
    'delay_risk': 0.20,  # lower historical delay rate = better
}

print('Setup complete. Weights:', EFFICIENCY_WEIGHTS)

In [ ]:
# ── Load and prepare master data ──────────────────────────────────────────────
df_trips  = pd.read_csv(TRIPS_PATH)
df_diesel = pd.read_csv(DIESEL_PATH)
df_tolls  = pd.read_csv(TOLLS_PATH)

# ── Parse toll rates ──────────────────────────────────────────────────────────
def parse_rates(s):
    try: return json.loads(s)
    except: return {}

rates_exp = df_tolls['rates'].apply(parse_rates).apply(pd.Series)
df_tolls  = pd.concat([df_tolls.drop(columns=['rates']), rates_exp], axis=1)
df_tolls['multiaxle_single'] = pd.to_numeric(df_tolls['multiaxle_single'], errors='coerce')
MEDIAN_TOLL_RATE = df_tolls['multiaxle_single'].median()

# ── Diesel lookup ─────────────────────────────────────────────────────────────
df_diesel['city_norm'] = df_diesel['city'].str.strip().str.title()
DIESEL_DICT = dict(zip(df_diesel['city_norm'], df_diesel['price']))
MEDIAN_DIESEL = df_diesel['price'].median()

def get_diesel(city): return DIESEL_DICT.get(str(city).title(), MEDIAN_DIESEL)

# ── Feature engineering ───────────────────────────────────────────────────────
trips = df_trips.rename(columns={'TRANSPORTATION_DISTANCE_IN_KM': 'distance_km'}).copy()
trips = trips[trips['distance_km'].notna() & (trips['distance_km'] > 0)].copy()

def extract_city(s):
    if pd.isna(s): return 'Unknown'
    parts = [p.strip() for p in str(s).split(',')]
    return parts[-2].title() if len(parts) >= 2 else parts[0].title()

def extract_cap(s):
    m = re.search(r'(\d+(?:\.\d+)?)\s*MT', str(s), re.I)
    return float(m.group(1)) if m and 1 <= float(m.group(1)) <= 40 else 20.0

def parse_latlon(s):
    try:
        p = str(s).split(',')
        return float(p[0].strip()), float(p[1].strip())
    except: return None, None

trips['origin_city']  = trips['Origin_Location'].apply(extract_city)
trips['dest_city']    = trips['Destination_Location'].apply(extract_city)
trips['lane']         = trips['OriginLocation_Code'].str.strip() + '_' + trips['DestinationLocation_Code'].str.strip()
trips['org_code']     = trips['OriginLocation_Code'].str.strip()
trips['des_code']     = trips['DestinationLocation_Code'].str.strip()
trips['load_tons']    = trips['vehicleType'].apply(extract_cap) * 0.80
trips['fuel_price']   = trips['origin_city'].apply(get_diesel)
trips['toll_cost']    = (trips['distance_km'] / 100).round() * MEDIAN_TOLL_RATE
trips['trip_start_dt'] = pd.to_datetime(trips['trip_start_date'], errors='coerce')
trips['planned_eta_dt'] = pd.to_datetime(trips['Planned_ETA'], errors='coerce')
trips['actual_eta_dt']  = pd.to_datetime(trips['actual_eta'],  errors='coerce')
trips['dwell_min']    = ((trips['actual_eta_dt'] - trips['planned_eta_dt'])
                         .dt.total_seconds() / 60).clip(lower=0).fillna(0)
trips['delay_flag']   = (~(trips['ontime'] == 'G')).astype(int)
trips['hour_of_day']  = trips['trip_start_dt'].dt.hour
trips['month']        = trips['trip_start_dt'].dt.month
trips['is_peak_hour'] = trips['hour_of_day'].apply(lambda h: 1 if (6<=h<10 or 17<=h<21) else 0)
trips['is_peak_season'] = trips['month'].apply(lambda m: 1 if m in [4,5,6] else 0)

# Parse lat/lon
trips[['org_lat','org_lon']] = trips['Org_lat_lon'].apply(lambda x: pd.Series(parse_latlon(x)))
trips[['des_lat','des_lon']] = trips['Des_lat_lon'].apply(lambda x: pd.Series(parse_latlon(x)))

print(f'Trips loaded: {len(trips):,} | Unique lanes: {trips["lane"].nunique()} | Unique origins: {trips["org_code"].nunique()}')

---
## 2. Lane Graph — the backbone of both features

Both route optimization and backhaul matching need a **lane graph**: a table of every known origin→destination pair with their historical performance metrics (average cost, travel time, delay rate).

In [ ]:
# ── Pricing function (from Module 1) ──────────────────────────────────────────
def compute_trip_price(row, params=None):
    if params is None: params = BEST_PARAMS
    dist   = float(row.get('distance_km', 0))
    load   = float(row.get('load_tons', params['truck_capacity_tons'] * 0.8))
    cap    = float(row.get('truck_capacity_tons', params['truck_capacity_tons']))
    diesel = float(row.get('fuel_price', MEDIAN_DIESEL))
    toll   = float(row.get('toll_cost', (dist/100) * MEDIAN_TOLL_RATE))
    dwell  = float(row.get('dwell_min', 0))

    fuel_cost   = dist * params['fuel_eff'] * diesel
    refrig_cost = (dist / params['avg_speed_kmh']) * params['refrig_rs_hr']
    driver_cost = (dist / params['km_per_driver_day']) * params['driver_cost_per_day']
    base_cost   = fuel_cost + refrig_cost + driver_cost + toll

    lane_pop   = float(row.get('lane_popularity', 0.5))
    alpha      = 1.0 + params['demand_weight'] * lane_pop
    load_util  = min(load / max(cap, 1), 1.0)
    beta       = 1.0 if load_util >= 0.5 else 0.80 + 0.40 * load_util
    idle_hrs   = max(0, (dwell - params['free_dwell_min']) / 60)
    gamma      = 1.0 + idle_hrs * params['dwell_penalty_rate']
    delta      = 1.0 + 0.10 * int(row.get('is_peak_hour',0)) + 0.08 * int(row.get('is_peak_season',0))

    trip_price = base_cost * alpha * beta * gamma * delta * (1 + params['margin'])
    return {
        'base_cost':       round(base_cost, 2),
        'trip_price_rs':   round(trip_price, 2),
        'price_per_ton_km': round(trip_price / max(dist * load, 1), 4),
        'alpha': round(alpha, 3), 'beta': round(beta, 3),
        'gamma': round(gamma, 3), 'delta': round(delta, 3),
    }


# ── Build lane performance graph ───────────────────────────────────────────────
lane_graph = (
    trips.groupby(['org_code', 'des_code', 'lane'])
    .agg(
        trip_count      = ('BookingID',     'count'),
        avg_distance_km = ('distance_km',   'mean'),
        avg_load_tons   = ('load_tons',     'mean'),
        avg_dwell_min   = ('dwell_min',     'mean'),
        delay_rate      = ('delay_flag',    'mean'),
        avg_fuel_price  = ('fuel_price',    'mean'),
        avg_toll_cost   = ('toll_cost',     'mean'),
        origin_city     = ('origin_city',   'first'),
        dest_city       = ('dest_city',     'first'),
        avg_org_lat     = ('org_lat',       'mean'),
        avg_org_lon     = ('org_lon',       'mean'),
        avg_des_lat     = ('des_lat',       'mean'),
        avg_des_lon     = ('des_lon',       'mean'),
    )
    .reset_index()
)

# Normalise popularity
lane_graph['lane_popularity'] = (
    np.log1p(lane_graph['trip_count']) /
    np.log1p(lane_graph['trip_count'].max())
)

# Estimate travel time (hours)
lane_graph['est_travel_hrs'] = lane_graph['avg_distance_km'] / BEST_PARAMS['avg_speed_kmh']
lane_graph['est_total_hrs']  = lane_graph['est_travel_hrs'] + lane_graph['avg_dwell_min'] / 60

# Price each lane
price_cols = lane_graph.apply(
    lambda r: pd.Series(compute_trip_price({
        'distance_km':   r['avg_distance_km'],
        'load_tons':     r['avg_load_tons'],
        'fuel_price':    r['avg_fuel_price'],
        'toll_cost':     r['avg_toll_cost'],
        'dwell_min':     r['avg_dwell_min'],
        'lane_popularity': r['lane_popularity'],
        'is_peak_hour':  0, 'is_peak_season': 0,
    })),
    axis=1
)
lane_graph = pd.concat([lane_graph, price_cols], axis=1)

print(f'Lane graph built: {len(lane_graph):,} lanes')
print(lane_graph[['org_code','des_code','avg_distance_km','trip_price_rs',
                   'delay_rate','est_total_hrs']].head(8).to_string())

---
## 3. Route Optimization

**Problem:** Given origin O and destination D, the truck may not have a direct lane. Find:
1. Direct route (if it exists)
2. All 1-stop intermediate routes (O → X → D)
3. Rank all options by cost, time, and a composite efficiency score
4. Return a ranked list the shipper can act on

In [ ]:
# ── Haversine distance (straight-line km between two lat/lon points) ───────────
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi  = math.radians(lat2 - lat1)
    dlam  = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))


# ── Route Optimizer ────────────────────────────────────────────────────────────
def optimize_routes(
    origin_code,
    dest_code,
    load_tons       = 15.0,
    max_stops       = 1,
    top_n           = 5,
    weights         = None,
    params          = None,
):
    """
    Find and rank all viable routes from origin_code to dest_code.

    Parameters
    ----------
    origin_code  : str  — OriginLocation_Code
    dest_code    : str  — DestinationLocation_Code
    load_tons    : float — cargo weight for this shipment
    max_stops    : int  — 0 = direct only, 1 = allow one intermediate hub
    top_n        : int  — number of options to return
    weights      : dict — cost/time/delay_risk weights for composite score
    params       : dict — pricing model parameters

    Returns
    -------
    DataFrame of ranked route options with full breakdown
    """
    if weights is None: weights = EFFICIENCY_WEIGHTS
    if params  is None: params  = BEST_PARAMS

    options = []

    # ── Lookup helper ─────────────────────────────────────────────────────────
    def get_lane(o, d):
        rows = lane_graph[(lane_graph['org_code']==o) & (lane_graph['des_code']==d)]
        return rows.iloc[0] if len(rows) > 0 else None

    def price_segment(lane_row, load):
        return compute_trip_price({
            'distance_km':    lane_row['avg_distance_km'],
            'load_tons':      load,
            'fuel_price':     lane_row['avg_fuel_price'],
            'toll_cost':      lane_row['avg_toll_cost'],
            'dwell_min':      lane_row['avg_dwell_min'],
            'lane_popularity':lane_row['lane_popularity'],
            'is_peak_hour':   0, 'is_peak_season': 0,
        }, params)

    # ── Option A: direct route ─────────────────────────────────────────────────
    direct = get_lane(origin_code, dest_code)
    if direct is not None:
        p = price_segment(direct, load_tons)
        options.append({
            'route_type':    'Direct',
            'stops':         f"{direct['origin_city']} → {direct['dest_city']}",
            'n_stops':       0,
            'total_dist_km': round(direct['avg_distance_km'], 1),
            'est_time_hrs':  round(direct['est_total_hrs'], 1),
            'trip_price_rs': p['trip_price_rs'],
            'price_per_ton_km': p['price_per_ton_km'],
            'delay_risk_pct': round(direct['delay_rate'] * 100, 1),
            'lane_segments': 1,
            'base_cost':     p['base_cost'],
            'org_lat': direct['avg_org_lat'], 'org_lon': direct['avg_org_lon'],
            'des_lat': direct['avg_des_lat'], 'des_lon': direct['avg_des_lon'],
            'via': None,
        })

    # ── Option B: 1-stop intermediate routes ──────────────────────────────────
    if max_stops >= 1:
        # All nodes reachable from origin
        from_origin = lane_graph[lane_graph['org_code'] == origin_code]['des_code'].unique()
        # All nodes that can reach destination
        to_dest     = lane_graph[lane_graph['des_code'] == dest_code]['org_code'].unique()
        # Intermediate hubs = intersection
        hubs        = set(from_origin) & set(to_dest)

        for hub in hubs:
            leg1 = get_lane(origin_code, hub)
            leg2 = get_lane(hub, dest_code)
            if leg1 is None or leg2 is None:
                continue

            p1 = price_segment(leg1, load_tons)
            p2 = price_segment(leg2, load_tons)

            total_dist  = leg1['avg_distance_km'] + leg2['avg_distance_km']
            total_time  = leg1['est_total_hrs']   + leg2['est_total_hrs'] + 2  # 2hr hub transfer
            total_price = p1['trip_price_rs'] + p2['trip_price_rs']
            # Combined delay risk = 1 - (1 - r1)(1 - r2) (independent risk)
            combined_delay = 1 - (1 - leg1['delay_rate']) * (1 - leg2['delay_rate'])

            hub_city = lane_graph[lane_graph['des_code'] == hub]['dest_city'].iloc[0] \
                       if len(lane_graph[lane_graph['des_code'] == hub]) > 0 else hub

            options.append({
                'route_type':    '1-Stop',
                'stops':         f"{leg1['origin_city']} → {hub_city} → {leg2['dest_city']}",
                'n_stops':       1,
                'total_dist_km': round(total_dist, 1),
                'est_time_hrs':  round(total_time, 1),
                'trip_price_rs': round(total_price, 2),
                'price_per_ton_km': round(total_price / max(total_dist * load_tons, 1), 4),
                'delay_risk_pct': round(combined_delay * 100, 1),
                'lane_segments': 2,
                'base_cost':     round(p1['base_cost'] + p2['base_cost'], 2),
                'org_lat': leg1['avg_org_lat'], 'org_lon': leg1['avg_org_lon'],
                'des_lat': leg2['avg_des_lat'], 'des_lon': leg2['avg_des_lon'],
                'via': hub_city,
            })

    if not options:
        print(f'No routes found from {origin_code} to {dest_code}')
        return pd.DataFrame()

    df = pd.DataFrame(options)

    # ── Composite efficiency score (lower is better for all three components) ──
    scaler = MinMaxScaler()
    metrics = df[['trip_price_rs', 'est_time_hrs', 'delay_risk_pct']].copy()
    # Handle edge case where all values are identical
    for col in metrics.columns:
        if metrics[col].nunique() == 1:
            metrics[col] = 0.5
    if len(df) > 1:
        scaled = scaler.fit_transform(metrics)
    else:
        scaled = np.array([[0.5, 0.5, 0.5]])

    df['score_cost']  = scaled[:, 0]
    df['score_time']  = scaled[:, 1]
    df['score_delay'] = scaled[:, 2]
    df['efficiency_score'] = (
        weights['cost']       * df['score_cost'] +
        weights['time']       * df['score_time'] +
        weights['delay_risk'] * df['score_delay']
    ).round(4)
    # Lower score = better (all components are costs)
    df['rank'] = df['efficiency_score'].rank(method='min').astype(int)
    df = df.sort_values('rank').head(top_n).reset_index(drop=True)

    return df


print('Route optimizer ready')

In [ ]:
# ── Run route optimization — example ─────────────────────────────────────────
# Use the most common origin in the dataset
EXAMPLE_ORIGIN = 'CHEORADMRCCB1'   # Kanchipuram (Daimler plant)
EXAMPLE_DEST   = 'CHEMMNFILCCA1'   # Chennai (Film Nagar — high-traffic dest)

route_options = optimize_routes(
    origin_code = EXAMPLE_ORIGIN,
    dest_code   = EXAMPLE_DEST,
    load_tons   = 14.0,
    max_stops   = 1,
    top_n       = 5,
)

print('=' * 70)
print('ROUTE OPTIONS — Kanchipuram → Chennai')
print('=' * 70)
if not route_options.empty:
    display_cols = ['rank', 'route_type', 'stops', 'total_dist_km',
                    'est_time_hrs', 'trip_price_rs', 'delay_risk_pct', 'efficiency_score']
    print(route_options[display_cols].to_string(index=False))
else:
    print('No routes found — trying different pair')
    # Fallback: use top lane pair from dataset
    top_lane = lane_graph.sort_values('trip_count', ascending=False).iloc[0]
    EXAMPLE_ORIGIN = top_lane['org_code']
    EXAMPLE_DEST   = top_lane['des_code']
    route_options  = optimize_routes(EXAMPLE_ORIGIN, EXAMPLE_DEST, load_tons=14.0)
    print(route_options[display_cols].to_string(index=False))

In [ ]:
# ── Visualise route options ────────────────────────────────────────────────────
def plot_route_options(route_df, title='Route Options Comparison'):
    if route_df.empty:
        print('No routes to plot'); return

    n = len(route_df)
    fig, axes = plt.subplots(1, 3, figsize=(15, max(4, n * 0.8 + 2)))
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)

    labels  = [f"#{r['rank']} {r['route_type']}" for _, r in route_df.iterrows()]
    colors  = ['#2ecc71' if i == 0 else '#3498db' if i == 1 else '#95a5a6'
               for i in range(n)]

    # Cost
    axes[0].barh(labels, route_df['trip_price_rs'], color=colors, edgecolor='white')
    axes[0].set_title('Trip Price (₹)', fontsize=11)
    axes[0].set_xlabel('₹')
    for i, v in enumerate(route_df['trip_price_rs']):
        axes[0].text(v * 0.01, i, f'₹{v:,.0f}', va='center', fontsize=9)
    axes[0].invert_yaxis()

    # Time
    axes[1].barh(labels, route_df['est_time_hrs'], color=colors, edgecolor='white')
    axes[1].set_title('Estimated Time (hours)', fontsize=11)
    axes[1].set_xlabel('Hours')
    for i, v in enumerate(route_df['est_time_hrs']):
        axes[1].text(v * 0.01, i, f'{v:.1f}h', va='center', fontsize=9)
    axes[1].invert_yaxis()

    # Efficiency score
    axes[2].barh(labels, route_df['efficiency_score'], color=colors, edgecolor='white')
    axes[2].set_title('Efficiency Score (lower = better)', fontsize=11)
    axes[2].set_xlabel('Score [0–1]')
    for i, v in enumerate(route_df['efficiency_score']):
        axes[2].text(v * 0.01, i, f'{v:.3f}', va='center', fontsize=9)
    axes[2].invert_yaxis()

    # Best route badge
    best = route_df.iloc[0]
    fig.text(0.5, -0.04,
             f'Recommended: {best["stops"]} | ₹{best["trip_price_rs"]:,.0f} | '
             f'{best["est_time_hrs"]:.1f}h | Delay risk {best["delay_risk_pct"]:.0f}%',
             ha='center', fontsize=10, color='#27ae60',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='#eafaf1', edgecolor='#27ae60'))

    plt.tight_layout()
    plt.savefig('route_options.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Saved: route_options.png')


plot_route_options(route_options, title=f'Route Options: {EXAMPLE_ORIGIN} → {EXAMPLE_DEST}')

In [ ]:
# ── Full ranked options table — the shipper-facing output ─────────────────────
def print_route_report(route_df):
    """Pretty-print the route recommendation report as a shipper would see it."""
    if route_df.empty:
        print('No routes available.'); return

    print('\n' + '═'*68)
    print('  ROUTE RECOMMENDATION REPORT — ColdChain Optimized')
    print('═'*68)

    for _, r in route_df.iterrows():
        badge = '★ BEST OPTION' if r['rank'] == 1 else f'Option #{int(r["rank"])}'
        print(f'\n  {badge}')
        print(f'  Route      : {r["stops"]}')
        print(f'  Type       : {r["route_type"]} ({r["lane_segments"]} leg(s))')
        print(f'  Distance   : {r["total_dist_km"]:,.0f} km')
        print(f'  Est. time  : {r["est_time_hrs"]:.1f} hrs')
        print(f'  Price      : ₹{r["trip_price_rs"]:,.0f}')
        print(f'  ₹/ton-km   : ₹{r["price_per_ton_km"]:.2f}')
        print(f'  Delay risk : {r["delay_risk_pct"]:.0f}%')
        print(f'  Eff. score : {r["efficiency_score"]:.3f}  (lower is better)')
        if r['via']:
            print(f'  Via hub    : {r["via"]}')
        print('  ' + '─'*64)

    print('\n  Weight scheme: Cost 45% | Time 35% | Delay risk 20%')
    print('═'*68 + '\n')


print_route_report(route_options)

---
## 4. Backhaul Matching

**Problem:** After dropping off cargo at destination D, the truck is empty and needs to return. Instead of running empty (100% deadhead cost), find nearby cargo opportunities heading roughly back toward the origin (or toward any profitable destination).

**Backhaul match criteria:**
1. Pickup location is within `max_detour_km` of the current position (destination)
2. Return direction — heading angle within `direction_tolerance` degrees of the homeward bearing
3. Ranked by: net profit (backhaul revenue − extra fuel cost) and direction efficiency

In [ ]:
# ── Bearing calculation (direction from A to B in degrees) ────────────────────
def bearing_deg(lat1, lon1, lat2, lon2):
    """Compass bearing from point 1 to point 2 (0 = North, 90 = East)."""
    dlon = math.radians(lon2 - lon1)
    lat1r, lat2r = math.radians(lat1), math.radians(lat2)
    x = math.sin(dlon) * math.cos(lat2r)
    y = math.cos(lat1r)*math.sin(lat2r) - math.sin(lat1r)*math.cos(lat2r)*math.cos(dlon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360


def angle_diff(a, b):
    """Smallest angular difference between two bearings."""
    diff = abs(a - b) % 360
    return diff if diff <= 180 else 360 - diff


# ── Backhaul Matcher ───────────────────────────────────────────────────────────
def find_backhaul_matches(
    current_location_code,   # truck is currently at this node
    home_location_code,      # operator's home/depot
    available_cargo_df,      # DataFrame of pending shipments (or use lane_graph)
    max_detour_km     = 150, # max extra km to reach pickup
    direction_tol_deg = 90,  # ±degrees from homeward bearing
    min_load_tons     = 3.0, # ignore micro-loads
    top_n             = 8,
    params            = None,
):
    """
    Find cargo opportunities for the empty truck at current_location_code.

    Parameters
    ----------
    current_location_code : str — where the truck is now (just delivered)
    home_location_code    : str — operator's home depot (defines "return direction")
    available_cargo_df    : DataFrame — pending shipments with org_code, des_code,
                            avg_distance_km, avg_load_tons etc. (use lane_graph as proxy)
    max_detour_km         : float — max extra km to reach pickup vs direct home run
    direction_tol_deg     : float — bearing tolerance toward home
    min_load_tons         : float — minimum load worth considering
    top_n                 : int   — number of matches to return
    params                : dict  — pricing parameters

    Returns
    -------
    DataFrame of ranked backhaul opportunities
    """
    if params is None: params = BEST_PARAMS

    # Get current position coordinates
    cur_rows = lane_graph[lane_graph['org_code'] == current_location_code]
    if cur_rows.empty:
        cur_rows = lane_graph[lane_graph['des_code'] == current_location_code]
        lat_col, lon_col = 'avg_des_lat', 'avg_des_lon'
    else:
        lat_col, lon_col = 'avg_org_lat', 'avg_org_lon'

    if cur_rows.empty:
        print(f'Location {current_location_code} not found in lane graph')
        return pd.DataFrame()

    cur_lat = cur_rows.iloc[0][lat_col]
    cur_lon = cur_rows.iloc[0][lon_col]

    # Get home coordinates
    home_rows = lane_graph[lane_graph['org_code'] == home_location_code]
    if home_rows.empty:
        home_lat, home_lon = cur_lat - 2, cur_lon - 2  # fallback offset
    else:
        home_lat = home_rows.iloc[0]['avg_org_lat']
        home_lon = home_rows.iloc[0]['avg_org_lon']

    # Bearing from current position back home
    home_bearing = bearing_deg(cur_lat, cur_lon, home_lat, home_lon)

    # Empty truck running cost (fuel + driver, no cargo revenue)
    direct_home_dist = haversine_km(cur_lat, cur_lon, home_lat, home_lon) * 1.25  # road factor
    empty_run_cost   = direct_home_dist * params['fuel_eff'] * MEDIAN_DIESEL

    matches = []

    for _, cargo in available_cargo_df.iterrows():
        # Skip tiny loads
        if cargo.get('avg_load_tons', 0) < min_load_tons:
            continue

        pickup_lat = cargo.get('avg_org_lat')
        pickup_lon = cargo.get('avg_org_lon')
        dropoff_lat = cargo.get('avg_des_lat')
        dropoff_lon = cargo.get('avg_des_lon')

        if any(pd.isna(x) for x in [pickup_lat, pickup_lon, dropoff_lat, dropoff_lon]):
            continue

        # Distance from truck's current position to cargo pickup
        detour_km = haversine_km(cur_lat, cur_lon, pickup_lat, pickup_lon) * 1.25
        if detour_km > max_detour_km:
            continue

        # Direction check — is the cargo heading roughly homeward?
        cargo_bearing = bearing_deg(pickup_lat, pickup_lon, dropoff_lat, dropoff_lon)
        dir_diff      = angle_diff(cargo_bearing, home_bearing)
        if dir_diff > direction_tol_deg:
            continue

        # Price the backhaul trip
        bh_price = compute_trip_price({
            'distance_km':    cargo['avg_distance_km'],
            'load_tons':      cargo['avg_load_tons'],
            'fuel_price':     cargo.get('avg_fuel_price', MEDIAN_DIESEL),
            'toll_cost':      cargo.get('avg_toll_cost', 0),
            'dwell_min':      cargo.get('avg_dwell_min', 0),
            'lane_popularity':cargo.get('lane_popularity', 0.3),
            'is_peak_hour':   0, 'is_peak_season': 0,
        }, params)

        # Detour cost (extra fuel to reach pickup)
        detour_fuel_cost = detour_km * params['fuel_eff'] * MEDIAN_DIESEL

        # Net profit vs running empty home
        net_profit = bh_price['trip_price_rs'] - detour_fuel_cost - empty_run_cost

        # Utilisation factor — what % of truck capacity is used
        util_pct = min(cargo['avg_load_tons'] / params['truck_capacity_tons'], 1.0) * 100

        matches.append({
            'cargo_lane':        cargo['lane'],
            'pickup_city':       cargo['origin_city'],
            'dropoff_city':      cargo['dest_city'],
            'detour_km':         round(detour_km, 1),
            'cargo_dist_km':     round(cargo['avg_distance_km'], 1),
            'load_tons':         round(cargo['avg_load_tons'], 1),
            'util_pct':          round(util_pct, 1),
            'backhaul_price_rs': bh_price['trip_price_rs'],
            'detour_cost_rs':    round(detour_fuel_cost, 2),
            'net_profit_rs':     round(net_profit, 2),
            'delay_risk_pct':    round(cargo['delay_rate'] * 100, 1),
            'direction_match_deg': round(dir_diff, 1),
            'est_time_hrs':      round(cargo['est_total_hrs'] + detour_km / params['avg_speed_kmh'], 1),
        })

    if not matches:
        print(f'No backhaul matches found within {max_detour_km}km and {direction_tol_deg}° tolerance')
        return pd.DataFrame()

    df = pd.DataFrame(matches)

    # Rank: highest net profit first, break ties by detour distance
    df = df.sort_values(['net_profit_rs', 'detour_km'],
                         ascending=[False, True]).head(top_n).reset_index(drop=True)
    df.insert(0, 'rank', range(1, len(df)+1))

    return df


print('Backhaul matcher ready')

In [ ]:
# ── Run backhaul matching — example ───────────────────────────────────────────
# Truck just delivered at CHEMMNFILCCA1 (Chennai), home depot is GURJMATVSHUA1 (Gurgaon)

CURRENT_POS  = 'CHEMMNFILCCA1'   # Chennai — just delivered
HOME_DEPOT   = 'GURJMATVSHUA1'   # Gurgaon — operator home

backhaul_matches = find_backhaul_matches(
    current_location_code = CURRENT_POS,
    home_location_code    = HOME_DEPOT,
    available_cargo_df    = lane_graph,   # in production: live pending orders table
    max_detour_km         = 200,
    direction_tol_deg     = 120,          # broad tolerance since dataset is limited
    min_load_tons         = 3.0,
    top_n                 = 8,
)

if not backhaul_matches.empty:
    print('BACKHAUL OPPORTUNITIES — Truck at Chennai, heading toward Gurgaon')
    print(backhaul_matches.to_string(index=False))
else:
    # Relax constraints and retry
    print('Relaxing detour constraint to 500km...')
    backhaul_matches = find_backhaul_matches(
        CURRENT_POS, HOME_DEPOT, lane_graph,
        max_detour_km=500, direction_tol_deg=150, top_n=8
    )
    if not backhaul_matches.empty:
        print(backhaul_matches.to_string(index=False))

In [ ]:
# ── Backhaul report — the driver-facing output ────────────────────────────────
def print_backhaul_report(bh_df, current_loc, home_loc):
    if bh_df.empty:
        print('No backhaul matches found.'); return

    print('\n' + '═'*68)
    print('  BACKHAUL MATCH REPORT — ColdChain Optimized')
    print(f'  Truck at: {current_loc}  |  Home depot: {home_loc}')
    print('═'*68)

    for _, r in bh_df.iterrows():
        badge = '★ BEST MATCH' if r['rank'] == 1 else f'Match #{int(r["rank"])}'
        prof_tag = '(PROFITABLE)' if r['net_profit_rs'] > 0 else '(marginal)'
        print(f'\n  {badge}  {prof_tag}')
        print(f'  Cargo     : {r["pickup_city"]} → {r["dropoff_city"]}')
        print(f'  Detour    : {r["detour_km"]:,.0f} km to reach pickup')
        print(f'  Cargo dist: {r["cargo_dist_km"]:,.0f} km  |  Load: {r["load_tons"]} MT  ({r["util_pct"]:.0f}% utilisation)')
        print(f'  Revenue   : ₹{r["backhaul_price_rs"]:,.0f}')
        print(f'  Detour cost: ₹{r["detour_cost_rs"]:,.0f}')
        print(f'  Net profit : ₹{r["net_profit_rs"]:,.0f}')
        print(f'  Est. time  : {r["est_time_hrs"]:.1f} hrs  |  Delay risk: {r["delay_risk_pct"]:.0f}%')
        print(f'  Direction  : {r["direction_match_deg"]:.0f}° off homeward bearing')
        print('  ' + '─'*64)

    total_vs_empty = bh_df.iloc[0]['backhaul_price_rs']
    print(f'\n  Best match earns ₹{total_vs_empty:,.0f} vs ₹0 running empty.')
    print('═'*68 + '\n')


print_backhaul_report(backhaul_matches, CURRENT_POS, HOME_DEPOT)

In [ ]:
# ── Visualise backhaul opportunities ──────────────────────────────────────────
def plot_backhaul_opportunities(bh_df, title='Backhaul Match Comparison'):
    if bh_df.empty:
        print('Nothing to plot'); return

    n = min(len(bh_df), 8)
    df = bh_df.head(n).copy()
    labels = [f"#{r['rank']} {r['pickup_city']}→{r['dropoff_city']}" for _, r in df.iterrows()]
    colors = ['#27ae60' if r['net_profit_rs'] > 0 else '#e74c3c' for _, r in df.iterrows()]

    fig, axes = plt.subplots(1, 3, figsize=(16, max(4, n*0.75+2)))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    # Net profit
    bars = axes[0].barh(labels, df['net_profit_rs'], color=colors, edgecolor='white')
    axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
    axes[0].set_title('Net Profit (₹)', fontsize=11)
    axes[0].set_xlabel('₹')
    for i, v in enumerate(df['net_profit_rs']):
        axes[0].text(max(v, 0) + 50, i, f'₹{v:,.0f}', va='center', fontsize=8)
    axes[0].invert_yaxis()

    # Detour distance
    axes[1].barh(labels, df['detour_km'], color='#3498db', edgecolor='white')
    axes[1].set_title('Detour to Pickup (km)', fontsize=11)
    axes[1].set_xlabel('km')
    for i, v in enumerate(df['detour_km']):
        axes[1].text(v * 0.01, i, f'{v:.0f}km', va='center', fontsize=8)
    axes[1].invert_yaxis()

    # Load utilisation
    axes[2].barh(labels, df['util_pct'], color='#9b59b6', edgecolor='white')
    axes[2].axvline(50, color='orange', linewidth=1, linestyle='--', label='50% threshold')
    axes[2].set_title('Truck Utilisation (%)', fontsize=11)
    axes[2].set_xlabel('%')
    axes[2].set_xlim(0, 105)
    axes[2].legend(fontsize=8)
    for i, v in enumerate(df['util_pct']):
        axes[2].text(v + 0.5, i, f'{v:.0f}%', va='center', fontsize=8)
    axes[2].invert_yaxis()

    profit_patch = mpatches.Patch(color='#27ae60', label='Net profitable')
    loss_patch   = mpatches.Patch(color='#e74c3c', label='Net loss')
    axes[0].legend(handles=[profit_patch, loss_patch], fontsize=8)

    plt.tight_layout()
    plt.savefig('backhaul_matches.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Saved: backhaul_matches.png')


plot_backhaul_opportunities(backhaul_matches,
    title=f'Backhaul Opportunities from {CURRENT_POS}')

---
## 5. Combined KPI Impact — Route + Backhaul

Measure how route optimization and backhaul matching together improve system-level metrics.

In [ ]:
# ── Simulate backhaul savings across all top delivery endpoints ────────────────
def simulate_backhaul_savings(n_simulations=200, max_detour=300, direction_tol=120):
    """
    For each of n randomly sampled completed trips, check if a backhaul
    match exists. Track: match rate, avg net profit, avg utilisation.
    """
    np.random.seed(42)
    results = []

    # Sample unique destination nodes as "truck just delivered here"
    dest_nodes = lane_graph['des_code'].unique()
    home_nodes = lane_graph['org_code'].unique()

    for _ in range(n_simulations):
        cur  = np.random.choice(dest_nodes)
        home = np.random.choice(home_nodes)
        if cur == home:
            continue

        bh = find_backhaul_matches(
            cur, home, lane_graph,
            max_detour_km=max_detour,
            direction_tol_deg=direction_tol,
            top_n=1
        )

        if bh.empty:
            results.append({'matched': 0, 'net_profit': 0, 'util_pct': 0})
        else:
            best = bh.iloc[0]
            results.append({
                'matched':    1,
                'net_profit': best['net_profit_rs'],
                'util_pct':   best['util_pct'],
            })

    df = pd.DataFrame(results)
    match_rate  = df['matched'].mean() * 100
    avg_profit  = df[df['matched']==1]['net_profit'].mean()
    avg_util    = df[df['matched']==1]['util_pct'].mean()
    profitable  = (df[df['matched']==1]['net_profit'] > 0).mean() * 100

    print('\n' + '='*55)
    print('  BACKHAUL SIMULATION RESULTS')
    print('='*55)
    print(f'  Simulations         : {n_simulations}')
    print(f'  Match rate          : {match_rate:.1f}%')
    print(f'  Avg net profit      : ₹{avg_profit:,.0f} per matched trip')
    print(f'  % profitable matches: {profitable:.1f}%')
    print(f'  Avg utilisation     : {avg_util:.1f}%')
    print('='*55)

    return df


bh_sim = simulate_backhaul_savings(n_simulations=300, max_detour=400)

In [ ]:
# ── Route optimization impact: best vs worst route price comparison ────────────
def simulate_route_savings(n_simulations=200):
    """
    For n random O→D pairs, compare best route price vs worst route price.
    Measures how much route optimization saves vs a naive (first found) choice.
    """
    np.random.seed(0)
    savings = []

    org_codes = lane_graph['org_code'].unique()
    des_codes = lane_graph['des_code'].unique()

    for _ in range(n_simulations):
        o = np.random.choice(org_codes)
        d = np.random.choice(des_codes)
        if o == d:
            continue

        opts = optimize_routes(o, d, load_tons=14.0, max_stops=1, top_n=10)
        if opts.empty or len(opts) < 2:
            continue

        best_price  = opts['trip_price_rs'].min()
        worst_price = opts['trip_price_rs'].max()
        saving_pct  = (worst_price - best_price) / worst_price * 100

        savings.append({
            'best_price':  best_price,
            'worst_price': worst_price,
            'saving_rs':   worst_price - best_price,
            'saving_pct':  saving_pct,
            'n_options':   len(opts),
        })

    df = pd.DataFrame(savings)
    print('\n' + '='*55)
    print('  ROUTE OPTIMIZATION SAVINGS SIMULATION')
    print('='*55)
    print(f'  Pairs with 2+ route options : {len(df)}')
    print(f'  Avg saving vs worst route   : {df["saving_pct"].mean():.1f}%')
    print(f'  Median saving               : ₹{df["saving_rs"].median():,.0f}')
    print(f'  Max saving found            : {df["saving_pct"].max():.1f}%')
    print('='*55)

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(df['saving_pct'], bins=30, color='steelblue', edgecolor='white')
    axes[0].axvline(df['saving_pct'].mean(), color='red', linestyle='--',
                    label=f'Mean {df["saving_pct"].mean():.1f}%')
    axes[0].set_title('Route Optimization Savings (%)')
    axes[0].set_xlabel('Saving vs naive route (%)')
    axes[0].legend()

    axes[1].scatter(df['n_options'], df['saving_pct'], alpha=0.4, s=18, color='darkorange')
    axes[1].set_title('More options = bigger potential saving')
    axes[1].set_xlabel('Number of route options available')
    axes[1].set_ylabel('Saving vs worst option (%)')

    plt.tight_layout()
    plt.savefig('route_savings_simulation.png', dpi=130, bbox_inches='tight')
    plt.show()

    return df


route_sim = simulate_route_savings(n_simulations=300)

In [ ]:
# ── Final combined KPI summary ─────────────────────────────────────────────────
backhaul_match_rate   = bh_sim['matched'].mean() * 100
backhaul_avg_profit   = bh_sim[bh_sim['matched']==1]['net_profit'].mean()
route_avg_saving      = route_sim['saving_pct'].mean() if not route_sim.empty else 0

print('\n' + '='*65)
print('  MODULE 2 COMBINED KPI SUMMARY — Route Opt + Backhaul Matching')
print('='*65)
print(f'  Backhaul match rate          : {backhaul_match_rate:.1f}% of empty trucks matched')
print(f'  Avg backhaul net profit      : ₹{backhaul_avg_profit:,.0f} per matched trip')
print(f'  Route optimization saving    : {route_avg_saving:.1f}% vs naive route selection')
print(f'  Efficiency score weights     : Cost {EFFICIENCY_WEIGHTS["cost"]*100:.0f}% | '
      f'Time {EFFICIENCY_WEIGHTS["time"]*100:.0f}% | '
      f'Delay {EFFICIENCY_WEIGHTS["delay_risk"]*100:.0f}%')
print('='*65)